# The Ultimate Guide to Zeyro ML Pipeline

Welcome to the production ML pipeline! This notebook is the definitive guide to generating data, tuning hyperparameters via Optuna, training the Behavioural Finance Score (BFS) and Repayment Propensity Score (RPS) models, and evaluating their metrics and explainability (SHAP).

In [1]:
import os
import sys
import numpy as np
import pandas as pd
from IPython.display import display
import logging

# Include project paths
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../')))

from python.scoring.fixtures import generate_random_fv
from python.scoring.scoring import compute_rps
from python.training.train_bfs import run as train_bfs
from python.training.train_rps import run as train_rps
from python.training.optuna_tuner import tune_bfs, tune_rps
from python.training.features import BFS_FEATURES, RPS_FEATURES, BFS_TARGET, RPS_TARGET
from sklearn.model_selection import train_test_split


c:\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Dataset Generation
We generate a synthetic dataset of 10,000 rows. The target variable is defaulted_within_90d.

In [2]:
def generate_training_data(n_samples=10000):
    print(f"Generating {n_samples} synthetic samples...")
    records = []
    all_features = set(BFS_FEATURES + RPS_FEATURES)
    
    for i in range(n_samples):
        fv = generate_random_fv(seed=i)
        rps_output = compute_rps(fv)
        pd_val = 1.0 - rps_output.probability
        
        record = {feat: fv.get(feat, 0.0) for feat in all_features}
        np.random.seed(i)
        is_default = int(np.random.rand() < pd_val)
        
        record[BFS_TARGET] = is_default
        if BFS_TARGET != RPS_TARGET:
             record[RPS_TARGET] = is_default
        records.append(record)
        
    df = pd.DataFrame(records)
    return df

df = generate_training_data(10000)
print(f"Target '{BFS_TARGET}' Default Rate: {df[BFS_TARGET].mean()*100:.2f}%")
display(df.head())


Generating 10000 synthetic samples...
Target 'defaulted_within_90d' Default Rate: 37.88%


,income.avg_monthly_credit_inr,quality.data_coverage_days,quality.source_diversity_code,cashflow.avg_days_to_near_zero,savings.recurring_sip_detected,emi.total_monthly_exposure_inr,quality.data_gap_count,income.income_regularity_score,network.p2p_transfer_ratio,cashflow.end_of_month_stress_score,...,emi.emi_to_income_ratio,emi.loan_stacking_signals,emi.bnpl_activity_detected,cashflow.min_balance_30d_inr,cashflow.balance_trend_slope,income.credit_to_debit_ratio,volatility.sudden_behavior_change_score,income.income_trend_90d,emi.missed_emi_signals_count,defaulted_within_90d
0,76264.514719,70.089192,0.0,3.920335,1.0,9994.054119,3.0,0.845857,0.0,0.131030,...,0.122718,1.0,0.0,8740.019526,0.004724,1.473105,0.134311,0.265465,0.0,0
1,28706.776162,83.734524,2.0,7.695130,0.0,27305.551768,3.0,0.775244,0.0,0.398813,...,0.210046,1.0,1.0,4453.742643,-0.097692,1.124507,0.039439,-0.072501,3.0,1
2,36560.997377,87.295987,0.0,4.964224,1.0,7497.053032,3.0,0.603695,0.0,0.164518,...,0.191600,1.0,0.0,10253.454532,-0.045982,0.654531,0.125534,-0.145376,0.0,1
3,62514.000705,45.478874,1.0,1.899341,0.0,25004.313600,2.0,0.374394,0.0,0.256647,...,0.492640,0.0,1.0,17323.992016,-0.064127,1.291290,0.332148,0.145923,2.0,1
4,48427.251796,30.432680,0.0,25.784802,0.0,15251.171260,2.0,0.753858,0.0,0.122978,...,0.028008,0.0,0.0,17613.009952,0.064221,1.330982,0.067797,-0.124784,0.0,0


## 2. Train/Validate/Test Splits
The pipelines natively implement a stratified split.

In [3]:
df_temp, df_test = train_test_split(df, test_size=0.15, stratify=df[BFS_TARGET], random_state=42)
df_train, df_val = train_test_split(df_temp, test_size=0.17647, stratify=df_temp[BFS_TARGET], random_state=42)

split_df = pd.DataFrame({
    "Dataset": ["Train", "Validation", "Test"],
    "Rows": [len(df_train), len(df_val), len(df_test)],
    "Percentage": [f"{len(df_train)/len(df)*100:.1f}%", f"{len(df_val)/len(df)*100:.1f}%", f"{len(df_test)/len(df)*100:.1f}%"]
})
display(split_df)


,Dataset,Rows,Percentage
0,Train,7000,70.0%
1,Validation,1500,15.0%
2,Test,1500,15.0%


## 3. Optuna Hyperparameter Tuning (BFS Model)
Optuna conducts a Bayesian search to find optimal hyperparameters.

In [4]:
# To keep this notebook fast, we use n_trials=5. In production, use 50-100.
best_params_bfs = tune_bfs(df, n_trials=5, run_name="bfs_optuna_demo")
print("\n*** BEST PARAMETERS FOUND BY OPTUNA FOR BFS ***")
for k, v in best_params_bfs.items():
    print(f"  {k}: {v}")


 BFS TRAINING RUN | run=bfs_optuna_demo_trial_000 | 2026-07-03T03:55:03Z
[SETUP]  Rows: 10000 | Target rate: 37.88%
[PARAMS] n_estimators=500 | learning_rate=0.0108 | max_depth=8 | min_child_weight=1 | subsample=0.5189 | colsample_bytree=0.9234 | reg_alpha=0.6918 | reg_lambda=0.0018
[SPLIT]  Train: 7000 | Val: 3000 (70/30)
────────────────────────────────────────────────────────────────────────────────
[TRAIN   ] Starting...
C:\Users\Admin\AppData\Roaming\Python\Python313\site-packages\xgboost\callback.py:386: UserWarning: [09:25:04] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()
[TRAIN   ] Done in 1.6s
[TRAIN]  Best round: 33
────────────────────────────────────────────────────────────────────────────────
[METRICS - TRAIN]
  AUC: 0.8697  KS: 0.5739  GINI: 0.7394  BRIER: 0.2060  AVERAGE_PRECISION: 0.8293
  Threshold=0.50  ACCURACY: 0.6281  PRECISION: 1.0000  RECA


*** BEST PARAMETERS FOUND BY OPTUNA FOR BFS ***
  n_estimators: 400
  learning_rate: 0.03645539920606815
  max_depth: 4
  min_child_weight: 8
  subsample: 0.606987495736368
  colsample_bytree: 0.8052384560854454
  reg_alpha: 1.2085595388604156
  reg_lambda: 0.053651285593500075
  objective: binary:logistic
  eval_metric: auc
  random_state: 42
  n_jobs: -1


## 4. Final BFS Training & Evaluation
Train using optimal parameters.

In [5]:
bfs_result = train_bfs(df, run_name="bfs_prod_v1", xgb_params=best_params_bfs)
print(f"\nArtifacts saved to: {bfs_result['model_path']}")

def display_metrics(result_dict):
    train_m = {k: v for k, v in result_dict['metrics']['train'].items() if not isinstance(v, dict)}
    val_m = {k: v for k, v in result_dict['metrics']['val'].items() if not isinstance(v, dict)}
    test_m = {k: v for k, v in result_dict['metrics']['test'].items() if not isinstance(v, dict)}
    m_df = pd.DataFrame([train_m, val_m, test_m], index=["Train", "Validation", "Test"])
    display(m_df.T)

print("\n--- BFS Core Evaluation Metrics ---")
display_metrics(bfs_result)

print("\n--- Top BFS Features by Global SHAP (Train Set) ---")
display(pd.DataFrame(bfs_result["shap"]["global_train"]))

print("\n--- Top BFS Features by Global SHAP (Test Set) ---")
display(pd.DataFrame(bfs_result["shap"]["global_test"]))


 BFS TRAINING RUN | run=bfs_prod_v1 | 2026-07-03T03:55:20Z
[SETUP]  Rows: 10000 | Target rate: 37.88%
[PARAMS] n_estimators=400 | learning_rate=0.0365 | max_depth=4 | min_child_weight=8 | subsample=0.6070 | colsample_bytree=0.8052 | reg_alpha=1.2086 | reg_lambda=0.0537
[SPLIT]  Train: 7000 | Val: 3000 (70/30)
────────────────────────────────────────────────────────────────────────────────
[TRAIN   ] Starting...
C:\Users\Admin\AppData\Roaming\Python\Python313\site-packages\xgboost\callback.py:386: UserWarning: [09:25:20] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()
[TRAIN   ] Done in 0.2s
[TRAIN]  Best round: 26
────────────────────────────────────────────────────────────────────────────────
[METRICS - TRAIN]
  AUC: 0.7406  KS: 0.3621  GINI: 0.4812  BRIER: 0.2088  AVERAGE_PRECISION: 0.6339
  Threshold=0.50  ACCURACY: 0.6640  PRECISION: 0.7577  RECALL: 0.1663  F1


Artifacts saved to: d:\z-business\artifacts\bfs\bfs_prod_v1\model.json

--- BFS Core Evaluation Metrics ---


,Train,Validation,Test
model,bfs_train,bfs_val,bfs_test
threshold,0.5,0.5,0.5
auc,0.7406,0.686,0.705
ks,0.3621,0.277,0.3335
gini,0.4812,0.372,0.41
brier,0.2088,0.2165,0.2141
average_precision,0.6339,0.5584,0.5748
accuracy,0.664,0.648,0.6533
precision,0.7577,0.6613,0.669
recall,0.1663,0.1444,0.1673



--- Top BFS Features by Global SHAP (Train Set) ---


,feature,mean_shap,reason
0,cashflow.balance_trend_slope,0.278008,Negative balance trajectory
1,income.income_regularity_score,0.090224,Irregular income pattern
2,cashflow.end_of_month_stress_score,0.074474,Month-end cash flow stress
3,emi.missed_emi_signals_count,0.045982,Prior missed EMI signals
4,emi.loan_stacking_signals,0.025878,Multiple new loans in 30 days (stacking)
5,savings.savings_to_income_ratio,0.008202,Insufficient savings buffer
6,network.new_vpa_ratio_30d,0.007098,High proportion of transfers to new unknown co...
7,emi.emi_to_income_ratio,0.006899,High debt-to-income ratio
8,cashflow.min_balance_30d_inr,0.006160,None
9,income.avg_monthly_credit_inr,0.005996,None



--- Top BFS Features by Global SHAP (Test Set) ---


,feature,mean_shap,reason
0,cashflow.balance_trend_slope,0.277471,Negative balance trajectory
1,income.income_regularity_score,0.091957,Irregular income pattern
2,cashflow.end_of_month_stress_score,0.075392,Month-end cash flow stress
3,emi.missed_emi_signals_count,0.045541,Prior missed EMI signals
4,emi.loan_stacking_signals,0.025228,Multiple new loans in 30 days (stacking)
5,savings.savings_to_income_ratio,0.008082,Insufficient savings buffer
6,network.new_vpa_ratio_30d,0.007209,High proportion of transfers to new unknown co...
7,emi.emi_to_income_ratio,0.006552,High debt-to-income ratio
8,income.avg_monthly_credit_inr,0.005952,None
9,cashflow.min_balance_30d_inr,0.005904,None


## 5. Optuna Tuning & Training (RPS Model)
Now we run the RPS process.

In [6]:
best_params_rps = tune_rps(df, n_trials=5, run_name="rps_optuna_demo")
print("\n*** BEST PARAMETERS FOUND BY OPTUNA FOR RPS ***")
for k, v in best_params_rps.items():
    print(f"  {k}: {v}")

rps_result = train_rps(df, run_name="rps_prod_v1", xgb_params=best_params_rps)

print("\n--- RPS Core Evaluation Metrics ---")
display_metrics(rps_result)

print("\n--- Top RPS Features by Global SHAP (Train Set) ---")
display(pd.DataFrame(rps_result["shap"]["global_train"]))

print("\n--- Top RPS Features by Global SHAP (Test Set) ---")
display(pd.DataFrame(rps_result["shap"]["global_test"]))


 RPS TRAINING RUN | run=rps_optuna_demo_trial_000 | 2026-07-03T03:55:42Z
[SETUP]  Rows: 10000 | Target rate: 37.88%
[PARAMS] n_estimators=300 | learning_rate=0.0204 | max_depth=7 | min_child_weight=11 | subsample=0.6009 | colsample_bytree=0.6348 | reg_alpha=0.2986 | reg_lambda=2.2920
[SPLIT]  Train: 7000 | Val: 3000 (70/30)
────────────────────────────────────────────────────────────────────────────────
[TRAIN   ] Starting...
C:\Users\Admin\AppData\Roaming\Python\Python313\site-packages\xgboost\callback.py:386: UserWarning: [09:25:42] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()
[TRAIN   ] Done in 0.1s
[TRAIN]  Best round: 11
────────────────────────────────────────────────────────────────────────────────
[METRICS - TRAIN]
  AUC: 0.7575  KS: 0.3754  GINI: 0.5150  BRIER: 0.2250  AVERAGE_PRECISION: 0.6507
  Threshold=0.50  ACCURACY: 0.6211  PRECISION: 0.0000  REC


*** BEST PARAMETERS FOUND BY OPTUNA FOR RPS ***
  n_estimators: 300
  learning_rate: 0.02043817688355324
  max_depth: 7
  min_child_weight: 11
  subsample: 0.6009472862670099
  colsample_bytree: 0.6348446478944421
  reg_alpha: 0.2985668800476883
  reg_lambda: 2.2919951507022556
  objective: binary:logistic
  eval_metric: auc
  random_state: 42
  n_jobs: -1


────────────────────────────────────────────────────────────────────────────────
[METRICS - TRAIN]
  AUC: 0.7575  KS: 0.3754  GINI: 0.5150  BRIER: 0.2250  AVERAGE_PRECISION: 0.6507
  Threshold=0.50  ACCURACY: 0.6211  PRECISION: 0.0000  RECALL: 0.0000  F1: 0.0000
  TP: 0       TN: 4348    FP: 0       FN: 2652  
────────────────────────────────────────────────────────────────────────────────
[METRICS - VALIDATION]
  AUC: 0.6913  KS: 0.2799  GINI: 0.3826  BRIER: 0.2277  AVERAGE_PRECISION: 0.5756
  Threshold=0.50  ACCURACY: 0.6213  PRECISION: 0.0000  RECALL: 0.0000  F1: 0.0000
  TP: 0       TN: 932     FP: 0       FN: 568   
────────────────────────────────────────────────────────────────────────────────
[METRICS - TEST (HELD OUT)]
  AUC: 0.7020  KS: 0.3356  GINI: 0.4040  BRIER: 0.2274  AVERAGE_PRECISION: 0.5711
  Threshold=0.50  ACCURACY: 0.6213  PRECISION: 0.0000  RECALL: 0.0000  F1: 0.0000
  TP: 0       TN: 932     FP: 0       FN: 568   
─────────────────────────────────────────────────


--- RPS Core Evaluation Metrics ---


,Train,Validation,Test
model,rps_train,rps_val,rps_test
threshold,0.5,0.5,0.5
auc,0.7575,0.6913,0.702
ks,0.3754,0.2799,0.3356
gini,0.515,0.3826,0.404
brier,0.225,0.2277,0.2274
average_precision,0.6507,0.5756,0.5711
accuracy,0.6211,0.6213,0.6213
precision,0.0,0.0,0.0
recall,0.0,0.0,0.0



--- Top RPS Features by Global SHAP (Train Set) ---


,feature,mean_shap,reason
0,cashflow.balance_trend_slope,0.079976,Negative balance trajectory
1,income.income_regularity_score,0.027114,Irregular income pattern
2,cashflow.end_of_month_stress_score,0.022337,Month-end cash flow stress
3,emi.missed_emi_signals_count,0.017533,Prior missed EMI signals
4,cashflow.min_balance_30d_inr,0.010848,None
5,emi.loan_stacking_signals,0.006674,Multiple new loans in 30 days (stacking)
6,cashflow.avg_days_to_near_zero,0.005901,Rapid cash depletion after payday
7,emi.bnpl_activity_detected,0.000472,None



--- Top RPS Features by Global SHAP (Test Set) ---


,feature,mean_shap,reason
0,cashflow.balance_trend_slope,0.079873,Negative balance trajectory
1,income.income_regularity_score,0.027415,Irregular income pattern
2,cashflow.end_of_month_stress_score,0.022538,Month-end cash flow stress
3,emi.missed_emi_signals_count,0.017362,Prior missed EMI signals
4,cashflow.min_balance_30d_inr,0.010788,None
5,emi.loan_stacking_signals,0.006570,Multiple new loans in 30 days (stacking)
6,cashflow.avg_days_to_near_zero,0.005874,Rapid cash depletion after payday
7,emi.bnpl_activity_detected,0.000512,None


## Inference: Apply Models to DataFrame
Load the trained models from their artifacts and score the entire dataset to calculate the BFS Score and RPS probability.

In [ ]:
import xgboost as xgb
from python.training.features import prepare_features
from python.scoring.scoring import probability_to_bfs

# Prepare features
X_bfs = prepare_features(df, BFS_FEATURES)
X_rps = prepare_features(df, RPS_FEATURES)

# Load models and calculate probabilities
model_bfs = xgb.XGBClassifier()
model_bfs.load_model(bfs_result['model_path'])
df['predicted_bfs_prob'] = model_bfs.predict_proba(X_bfs)[:, 1]
df['bfs_score'] = df['predicted_bfs_prob'].apply(probability_to_bfs)

model_rps = xgb.XGBClassifier()
model_rps.load_model(rps_result['model_path'])
df['predicted_rps_prob'] = model_rps.predict_proba(X_rps)[:, 1]

# Display results
display_cols = ['predicted_bfs_prob', 'bfs_score', 'predicted_rps_prob']
# Include targets if they exist in the dataframe
if BFS_TARGET in df.columns:
    display_cols.insert(0, BFS_TARGET)

display(df[display_cols].head(10))